# 01 — Exploratory Data Analysis
Customer Churn Prediction Pipeline — IBM Telco Dataset

In [1]:
import sys, os
# Move to project root so relative paths (data/, outputs/) resolve correctly
# Works whether the kernel starts from notebooks/ or the project root.
_here = os.path.abspath('.')
if os.path.basename(_here) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.path.abspath('.'))
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display
from pathlib import Path

from src.data.loader import load_raw_data
from src.data.cleaner import clean_data, validate_clean_data
from src.visualization.plots import (
    plot_churn_distribution,
    plot_churn_by_contract,
    plot_tenure_distribution,
    plot_charges_boxplot,
)

FIGURES = Path('outputs/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Dataset Overview

In [2]:
df_raw = load_raw_data()
print('Shape:', df_raw.shape)
print('\nDtypes:')
print(df_raw.dtypes)
df_raw.head(3)

Shape: (7043, 21)

Dtypes:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [3]:
print('Missing values per column:')
print(df_raw.isnull().sum())
print('\nTotalCharges with empty string:',
      (df_raw['TotalCharges'].astype(str).str.strip() == '').sum())

Missing values per column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

TotalCharges with empty string: 11


## 2. Cleaning

In [4]:
df = clean_data(df_raw)
validate_clean_data(df)
print('Clean shape:', df.shape)
print('Churn rate: {:.1f}%'.format(df['Churn'].mean() * 100))

Clean shape: (7043, 20)
Churn rate: 26.5%


## 3. Target Distribution

In [5]:
path = plot_churn_distribution(df, FIGURES)
img = mpimg.imread(path)
fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Churn by Contract Type

In [6]:
path = plot_churn_by_contract(df, FIGURES)
img = mpimg.imread(path)
fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

by_contract = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
print('Churn rate by contract:')
print(by_contract.map('{:.1%}'.format))

Churn rate by contract:
Contract
Month-to-month    42.7%
One year          11.3%
Two year           2.8%
Name: Churn, dtype: str


## 5. Tenure Distribution

In [7]:
path = plot_tenure_distribution(df, FIGURES)
img = mpimg.imread(path)
fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

print('Median tenure — No Churn: {:.0f}m | Churn: {:.0f}m'.format(
    df.loc[df['Churn']==0, 'tenure'].median(),
    df.loc[df['Churn']==1, 'tenure'].median(),
))

Median tenure — No Churn: 38m | Churn: 10m


## 6. Monthly Charges Distribution

In [8]:
path = plot_charges_boxplot(df, FIGURES)
img = mpimg.imread(path)
fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

print('Median MonthlyCharges — No Churn: ${:.2f} | Churn: ${:.2f}'.format(
    df.loc[df['Churn']==0, 'MonthlyCharges'].median(),
    df.loc[df['Churn']==1, 'MonthlyCharges'].median(),
))

Median MonthlyCharges — No Churn: $64.43 | Churn: $79.65


## 7. Key Findings

1. **Overall churn rate is ~26.5%** — the dataset is imbalanced; accuracy alone is a misleading metric (a classifier that always predicts "No Churn" reaches 73.5% accuracy).

2. **Contract type is the strongest single predictor**: Month-to-month customers churn at ~42%, vs ~11% for one-year and ~3% for two-year contracts. Customers locked into longer commitments are far less likely to leave.

3. **New customers are most vulnerable**: Median tenure at churn is ~10 months vs ~38 months for retained customers. The first year is the critical retention window — onboarding quality directly impacts lifetime value.

4. **Churned customers pay more per month**: Median MonthlyCharges for churned customers (~$79) is significantly higher than for retained ones (~$65). High-spend customers without long-term commitment are the highest-risk segment.

5. **Customers with internet service (especially Fiber optic) churn more**: The combination of high monthly charges and no contract creates a segment where switching costs are near zero — the ideal target for proactive retention offers.